In [0]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

In [0]:
tickers = ["AMD", "NVDA", "MSFT", "TXN","MU"]

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/137.0.0.0 Safari/537.36"
    )
}

# Keep as datetime
ingestion_date = pd.Timestamp.now()

all_data = []

for ticker in tickers:
    url = f"https://finviz.com/stock?t={ticker}&ty=c&ta=1&p=d"

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    table = soup.find_all("table")[9]

    rows = [
        [cell.get_text(strip=True) for cell in tr.find_all(["th", "td"])]
        for tr in table.find_all("tr")
    ]

    rows = [row for row in rows if row][:7]

    df = pd.DataFrame(rows, columns=["Metrics", "Value"])

    df["Ticker"] = ticker
    df["Ingestion_Date"] = ingestion_date

    all_data.append(df)

final_df = pd.concat(all_data, ignore_index=True)

# Data type conversions
final_df = final_df.astype({
    "Metrics": "string",
    "Ticker": "string",
    "Value": "float64"
})

# Ensure datetime64[ns]
final_df["Ingestion_Date"] = pd.to_datetime(
    final_df["Ingestion_Date"]
)

display(final_df)

print("\nColumn data types:")
print(final_df.dtypes)

print("\nDetailed info:")
final_df.info()

spark_df = spark.createDataFrame(final_df)
spark_df.write.mode("append").format("delta").saveAsTable('workspace.finvizwebscrapping.01_bronze_layer')